# 00b · Transformers ladder check (Milestone 0 follow-up)

M0's retries fell back to vLLM because transformers ran out of memory on the T4s. This
notebook replays only M0's failed chunks through the fixed transformers ladder (64-token
window penalty, like Ollama) and compares each rung with M0's vLLM retries. It also prints
the exact JSON parse error of the failed primary output. M0's own results are not changed.

**Kaggle:** Accelerator **GPU T4 x2**, Internet **on**, secret `HF_TOKEN`. No vLLM needed.
Expected time: a few minutes of model load plus a few minutes per retry (not yet measured).

In [ ]:
REPO_URL = "https://github.com/z26qin/credit-extractor.git"
BRANCH = "m1-collect"

In [ ]:
import os, subprocess, sys, site
ROOT = "/kaggle/working" if os.path.exists("/kaggle/working") else (
    "/content" if os.path.exists("/content") else os.getcwd())
REPO = os.path.join(ROOT, "credit-extractor")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "-q", "--ff-only"], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
with open(os.path.join(site.getsitepackages()[0], "credit_extractor.pth"), "w") as f:
    f.write(REPO + "\n")      # `import src` survives a kernel restart
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

In [ ]:
# after a kernel restart, start again from this cell
from src.platform import setup
env = setup()
assert env.data_repo, "HF_TOKEN missing: attach it as a secret (M0 outputs are on the Hub)"
assert env.gpus, "no GPU visible: pick GPU T4 x2"

In [ ]:
from src import reproduce
cfg = reproduce.load_cfg()
plan = reproduce.prepare(env, cfg)      # pulls M0's records from the Hub
rows = reproduce.ladder_check(env, plan, cfg)

In [ ]:
import pandas as pd
display(pd.DataFrame(rows))
print(f"session wall time: {env.session_minutes():.1f} min")